# 03 — Evaluation harness

This notebook freezes the ruler before model comparison. It partitions truth,
physically separates development from holdout, defines a finite case-grouping
rule and tests one-to-one matching with deliberately difficult controls.

It does not train a detector or select a threshold.


## 1. Setup


In [ ]:
import importlib
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")
    or ("/content/drive/MyDrive/anomaly_detection" if IN_COLAB
        else Path.home() / "anomaly_detection_data")
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")  # or "petrobras_3w"
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_10_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_10_1_run2",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json

CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
EVAL_ROOT = RUN_ROOT / "SPEC-EVAL"
SPLIT_ROOT = RUN_ROOT / "SPLITS"

import evaluation_core
evaluation_core = importlib.reload(evaluation_core)
from evaluation_core import (
    ALERT_COLUMNS, EVALUATION_CORE_VERSION, CASE_COLUMNS,
    evaluate_alerts, evaluate_cases, form_cases, partition_truth, scores_to_alerts,
)

EDA_VERSION = EVALUATION_VERSION = "2.1.0"
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{EDA_VERSION}" / SECTOR / f"{SECTOR}_eda_v2_1_run1"
OUTPUT_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{EVALUATION_VERSION}" / SECTOR / f"{SECTOR}_evaluation_v2_1_run1"

POLICY = {
    "telecom": {
        "decision_horizon_seconds": 48 * 3600,
        "lead_time_hours": [6, 12, 24, 48],
        "exposure_unit": "entity_day",
        "false_case_budget": 0.01,
        "case_gap_seconds": 3600,
        "threshold_quantiles": [0.99, 0.995, 0.9975, 0.999, 0.9995, 0.9998],
    },
    "petrobras_3w": {
        "decision_horizon_seconds": 6 * 3600,
        "lead_time_hours": [0.5, 1, 3, 6],
        "exposure_unit": "episode",
        "false_case_budget": 0.10,
        "case_gap_seconds": 0,
        "threshold_quantiles": [0.995, 0.999, 0.9995, 0.9998, 0.9999, 0.99995],
    },
}


## 2. Partition truth and inspect denominators


In [ ]:
if not EVAL_ROOT.is_dir():
    raise FileNotFoundError("This source has no SPEC-EVAL")
manifest = read_json(CORE_ROOT / "manifest.json")
decisions = read_json(EDA_ROOT / "eda_decisions.json")
registry = pd.read_parquet(CORE_ROOT / "entity_registry.parquet")
events = pd.read_parquet(EVAL_ROOT / "fault_events.parquet")
intervals = pd.read_parquet(EVAL_ROOT / "fault_entity_intervals.parquet")
condition_path = EVAL_ROOT / "condition_states.parquet"
conditions = pd.read_parquet(condition_path) if condition_path.is_file() else pd.DataFrame()

time_path = SPLIT_ROOT / "time_partitions.parquet"
entity_path = SPLIT_ROOT / "entity_partitions.parquet"
time_partitions = pd.read_parquet(time_path) if time_path.is_file() else pd.DataFrame()
entity_partitions = pd.read_parquet(entity_path) if entity_path.is_file() else pd.DataFrame()
primary_split = "time" if not time_partitions.empty else "entity"

truth, truth_audit, truth_summary = partition_truth(
    events, intervals, conditions, registry,
    primary_split=primary_split,
    time_partitions=time_partitions,
    entity_partitions=entity_partitions,
)
display(truth_summary)
display(pd.Series({
    "development_faults": len(truth["development"]["fault_events"]),
    "holdout_faults_sealed": len(truth["holdout"]["fault_events"]),
}, name="count").to_frame())


## 3. Adversarial contract tests


In [ ]:
BASE = pd.Timestamp("2025-01-01", tz="UTC")
event_columns = ["fault_id", "fault_type", "domain_id", "onset_ts", "observable_ts",
                 "impact_ts", "end_ts", "group_id", "label_source", "source_instance_id"]
interval_columns = ["fault_id", "entity_id", "start_ts", "end_ts", "label_source", "source_instance_id"]
test_events = pd.DataFrame([
    ("F1", "fault_a", "asset-1", BASE, BASE, BASE + pd.Timedelta(minutes=30), BASE + pd.Timedelta(hours=2), None, "test", "one"),
    ("F2", "fault_b", "asset-1", BASE + pd.Timedelta(minutes=10), BASE + pd.Timedelta(minutes=10), pd.NaT, BASE + pd.Timedelta(hours=2), None, "test", "two"),
], columns=event_columns)
test_intervals = pd.DataFrame([
    ("F1", "asset-1", BASE, BASE + pd.Timedelta(hours=2), "test", "one"),
    ("F2", "asset-1", BASE + pd.Timedelta(minutes=10), BASE + pd.Timedelta(hours=2), "test", "two"),
], columns=interval_columns)

def alert(alert_id, model, entity, minute, score=2.0):
    start = BASE + pd.Timedelta(minutes=minute)
    return (alert_id, model, entity, "episode-1", start, start + pd.Timedelta(minutes=1),
            start, score, 1, "metric__level")

alerts = pd.DataFrame([
    alert("A1", "rapid", "asset-1", 15),
    alert("A2", "drift", "asset-1", 20),
], columns=ALERT_COLUMNS)

# Overlapping fault windows still receive maximum-cardinality one-to-one credit.
alert_result = evaluate_alerts(
    alerts, test_events, test_intervals,
    exposure_value=10, exposure_unit="entity_day", decision_horizon_seconds=3600,
)
assert alert_result["fault_results"]["detected"].sum() == 2

# Two channels on one entity become one operational case.
cases, members = form_cases(
    alerts, gap_seconds=600, thresholds={"rapid": 1.0, "drift": 1.0}
)
assert len(cases) == 1 and len(members) == 2

# A-B-C topology chains may not create an unresolved case.
chain_alerts = pd.DataFrame([
    alert("B1", "rapid", "a", 0),
    alert("B2", "rapid", "b", 1),
    alert("B3", "rapid", "c", 2),
], columns=ALERT_COLUMNS)
chain_groups = pd.DataFrame([
    ("a", "group", "AB"), ("b", "group", "AB"),
    ("b", "group", "BC"), ("c", "group", "BC"),
], columns=["entity_id", "group_type", "group_id"])
chain_cases, _ = form_cases(
    chain_alerts, chain_groups, gap_seconds=600, thresholds={"rapid": 1.0}
)
assert len(chain_cases) == 2
assert not ((chain_cases.scope_type != "entity") & chain_cases.scope_id.isna()).any()

# One case receives at most one event credit.
case_result = evaluate_cases(
    cases, members, test_events, test_intervals,
    exposure_value=10, exposure_unit="entity_day", decision_horizon_seconds=3600,
)
assert case_result["fault_results"]["detected"].sum() == 1
assert case_result["case_matches"].match_status.eq("matched").sum() == 1

# Dense point alerts must not receive more event credit than one alert.
one_event = test_events.iloc[[0]].copy()
one_interval = test_intervals.iloc[[0]].copy()
single = pd.DataFrame([alert("P1", "rapid", "asset-1", 15)], columns=ALERT_COLUMNS)
dense = pd.DataFrame([
    alert(f"P{index}", "rapid", "asset-1", minute)
    for index, minute in enumerate(range(15, 25), start=1)
], columns=ALERT_COLUMNS)
single_cases, single_members = form_cases(
    single, gap_seconds=600, thresholds={"rapid": 1.0}
)
dense_cases, dense_members = form_cases(
    dense, gap_seconds=600, thresholds={"rapid": 1.0}
)
single_result = evaluate_cases(
    single_cases, single_members, one_event, one_interval,
    exposure_value=10, exposure_unit="entity_day", decision_horizon_seconds=3600,
)
dense_result = evaluate_cases(
    dense_cases, dense_members, one_event, one_interval,
    exposure_value=10, exposure_unit="entity_day", decision_horizon_seconds=3600,
)
assert len(single_cases) == len(dense_cases) == 1
assert single_result["fault_results"].detected.sum() == dense_result["fault_results"].detected.sum() == 1

# The content-invariance proof runs in 01A/01B. Here we verify that the
# model input is the SPEC-CORE directory, not its truth-bearing sibling.
run_manifest = read_json(RUN_ROOT / "run_manifest.json")
assert run_manifest["evaluation_mounted"]
assert CORE_ROOT.name == "SPEC-CORE" and EVAL_ROOT.name == "SPEC-EVAL"
assert not any((CORE_ROOT / f"{name}.parquet").exists() for name in [
    "fault_events", "fault_entity_intervals", "condition_states"
])

tests = pd.DataFrame({
    "test": ["overlapping-window matching", "channel consolidation",
             "no topology chaining", "one case-one credit",
             "point-adjustment control", "SPEC-CORE truth boundary"],
    "status": "pass",
})
display(tests)


## 4. Freeze policy and physically separate holdout


In [ ]:
development_faults = len(truth["development"]["fault_events"])
one_fault_step = 1 / development_faults if development_faults else np.nan
z = 1.96
if development_faults:
    denominator = 1 + z ** 2 / development_faults
    centre = (0.5 + z ** 2 / (2 * development_faults)) / denominator
    half_width = z * np.sqrt(0.25 / development_faults + z ** 2 / (4 * development_faults ** 2)) / denominator
else:
    centre = half_width = np.nan

policy = {
    "evaluation_version": EVALUATION_VERSION,
    "evaluation_core_version": EVALUATION_CORE_VERSION,
    "sector": SECTOR,
    "canonical_fingerprint": manifest["fingerprint"],
    "primary_split": primary_split,
    **POLICY[SECTOR],
    "selection_channels": ["rapid_residual", "drift_cusum"],
    "cusum_allowance": 0.5,
    "budget_safety_factor": 0.90,
    "equivalence_margin": 0.02,
    "channel_persistence": {
        "rapid_residual": 2,
        "drift_cusum": 1,
        "dispersion_change": 2,
        "pca_spe": 2,
        "isolation_forest": 2,
    },
    "recovery_observations": 2,
    "matching": "same affected entity, common decision horizon, deterministic maximum-cardinality one-to-one credit",
    "case_rule": "finite time gap and common resolvable topology scope; no transitive chaining",
    "selection": "predeclared per-channel calibration-quantile grid; upper confidence bound inside 90% of case budget; simplest portfolio within equivalence margin",
    "development_fault_count": development_faults,
    "one_fault_recall_step": one_fault_step,
    "recall_wilson_interval_at_50_percent": [centre - half_width, centre + half_width],
    "evidence_limit": "small recall differences are not statistically resolvable; simplicity is the declared tie-break",
    "drift_failure_branch": [
        "test observable operating-regime conditioning",
        "test a longer drift horizon",
        "declare the fault class not detectable from available telemetry if neither helps",
    ],
    "validation_role": "primary real-data evidence" if SECTOR == "petrobras_3w" else "synthetic methodological testbed",
    "holdout_used": False,
}

with new_output_directory(OUTPUT_ROOT) as output:
    for partition, folder in (("development", "development"), ("holdout", "holdout_sealed")):
        target = output / folder; target.mkdir()
        for table_name, frame in truth[partition].items():
            if not frame.empty:
                frame.to_parquet(target / f"{table_name}.parquet", index=False)
    truth_audit.to_csv(output / "truth_partition_audit.csv", index=False)
    write_json(output / "evaluation_policy.json", policy)

display(pd.Series(policy, name="value").to_frame())
print("PASS — evaluation policy frozen before modelling")
print("Saved:", OUTPUT_ROOT)
print("Next: 04_SIMPLE_ANOMALY_MODELS.ipynb")
